# L08 · Features sin fuga causal

**S8 · Lectura: Cap. 9 · 40–55 min**

## La pregunta

¿La feature estaba disponible antes del tratamiento?

## Qué harás

Detectar y quitar variables posteriores al tratamiento.

- Lee la idea breve.
- Ejecuta la demostración y mira el gráfico.
- Completa las celdas marcadas.
- Escribe qué significa el resultado y qué no permite afirmar.

**Producto:** Una auditoría de features y un conjunto limpio para A2.

> **Idea clave:** Una feature muy predictiva puede ser información del futuro y falsear el efecto.

La solución está en `la ruta de rescate docente (no incluida)`. Intenta primero el notebook del estudiante.

In [ ]:
# Localiza lab_common.py (carpeta actual o una superior)
import sys, pathlib
for _p in [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3]]:
    if (_p / "lab_common.py").exists():
        sys.path.insert(0, str(_p)); break

## 0. Preparación

In [ ]:
from lab_common import *
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
rng = setup()
sc = load_scenario("synthetic/inference/ihdp_medium"); sc.card()
X, y, t = sc.xyt("train")
n = len(y); base = list(sc.covariates)

In [ ]:
# Vista rápida de los datos del escenario
import matplotlib.pyplot as plt
import numpy as np
_fig, _ax = plt.subplots(1, 2, figsize=(9, 3.2))
try:
    _scenario = globals().get("sc") or globals().get("lc")
    if _scenario is None:
        raise NameError("No hay un escenario cargado para el gráfico editorial")
    _df = _scenario.train
    if _scenario.treatment is not None and _scenario.outcome is not None:
        _t = _df[_scenario.treatment].to_numpy(float)
        _y = _df[_scenario.outcome].to_numpy(float)
        if np.unique(_t).size == 2:
            _counts = [int((_t == value).sum()) for value in (0, 1)]
            _ax[0].bar(["control", "tratado"], _counts)
            _ax[0].set_title("Asignación")
            _ax[1].boxplot([_y[_t == 0], _y[_t == 1]], labels=["control", "tratado"])
            _ax[1].set_title("Resultado observado")
        else:
            _ax[0].hist(_t, bins=20)
            _ax[0].set_title(_scenario.treatment)
            _ax[1].scatter(_t, _y, s=8, alpha=.35)
            _ax[1].set_title(f"{_scenario.treatment} frente a {_scenario.outcome}")
    elif _scenario.adj is not None:
        _ax[0].imshow(_scenario.adj, cmap="Greys", interpolation="nearest")
        _ax[0].set_title("Grafo del escenario")
        _ax[1].axis("off")
    else:
        _ax[0].axis("off"); _ax[1].axis("off")
    _fig.suptitle(f"Escenario: {_scenario.name}")
    _fig.tight_layout(); plt.show()
finally:
    plt.close(_fig)

## 1. Qué es una fuga *causal* (§9.1)
En predicción, fuga = usar información que no tendrás al predecir. En causalidad es más grave: una covariable **medida después del tratamiento** (o **calculada con $Y$**) no solo infla la métrica: **cambia el estimando**. Ajustar por un mediador bloquea parte del efecto; ajustar por una consecuencia de $Y$ lo contamina.

Construimos tres fugas con **nombres neutros** (nadie las anuncia con un `leak_`):
- `indice_a`: un seguimiento a 6 meses (depende de $Y$ y de $T$).
- `indice_b`: el promedio de $Y$ del caso más parecido y de la propia fila (un *target encoding* **calculado con todo el conjunto**).
- `indice_c`: una variable causada por el tratamiento.

In [ ]:
def con_fugas(X, y, t, rng):
    Z = (X - X.mean(0)) / X.std(0)
    vecinos = NearestNeighbors(n_neighbors=2).fit(Z[:, :3]).kneighbors(Z[:, :3], return_distance=False)
    a = 0.7 * y + 0.5 * t + rng.normal(0, .3, len(y))
    b = y[vecinos].mean(1)
    c = 0.8 * t + 0.3 * Z[:, 0] + rng.normal(0, .5, len(y))
    return pd.DataFrame({"indice_a": a, "indice_b": b, "indice_c": c})
fugas = con_fugas(X, y, t, np.random.default_rng(3))
df = pd.DataFrame(X, columns=base).join(fugas)
FUGAS = ["indice_a", "indice_b", "indice_c"]
print("columnas disponibles:", df.shape[1], "(", len(base), "limpias +", len(FUGAS), "con fuga)")

## 2. Auditoría por metadatos: rol y momento de medición (§9.2)
La primera defensa es **documentar** cuándo se mide cada columna y con qué se calcula. La función `auditar_features` marca lo que viola las reglas.

In [ ]:
metadatos = pd.DataFrame({"feature": base + FUGAS})
metadatos["medido_en"] = ["pre"] * len(base) + ["post", "pre", "post"]
metadatos["calculado_con"] = ["fila"] * len(base) + ["fila", "todo el conjunto (incluye Y)", "fila"]
def auditar_features(meta):
    """Marca features medidas tras el tratamiento o calculadas con información global que incluye Y."""
    m = meta.copy()
    m["viola"] = np.where(m.medido_en == "post", "posterior al tratamiento",
                 np.where(m.calculado_con.str.contains("incluye Y"), "derivada de Y", ""))
    return m[m.viola != ""][["feature", "viola"]]
show(auditar_features(metadatos))

**Mira el resultado.** `indice_b` figura como medida «pre». ¿Qué lo delata en los metadatos y qué pasa cuando los metadatos no existen?

**Respuesta:**

Lo delata **cómo se calculó** (con todo el conjunto, incluyendo Y), no cuándo se midió. Si los metadatos no existen o están mal, hace falta un detector **por contenido**: sección 4.



## 3. Lo que cambia: efecto estimado y métrica predictiva
`X_base` = covariables limpias; `X_ampliada` = base + 3 fugas. Mismo estimador (OLS con controles), mismos datos.

In [ ]:
def ate_ols(df, cols):
    r = ols(y, np.column_stack([t, df[cols].to_numpy()])); return r["coef"][1], r["se"][1]
def r2(df, cols):
    A = np.column_stack([np.ones(len(y)), t, df[cols].to_numpy()]); b = np.linalg.lstsq(A, y, rcond=None)[0]
    return 1 - ((y - A @ b) ** 2).sum() / ((y - y.mean()) ** 2).sum()
filas = []
for nombre, cols in [("X_base", base), ("X_ampliada (+3 fugas)", base + FUGAS)] + [(f"base + {f}", base + [f]) for f in FUGAS]:
    e, s = ate_ols(df, cols); lo, hi = ci95(e, s)
    filas.append({"covariables": nombre, "ATE": e, "IC inf": lo, "IC sup": hi, "R² del modelo": r2(df, cols)})
tab = pd.DataFrame(filas); show(tab.round(3))
e0 = tab.ATE[0]; e1 = tab.ATE[1]
print(f"ΔATE (ampliada − base) = {e1 - e0:+.3f}   |   ΔR² = {tab['R² del modelo'][1] - tab['R² del modelo'][0]:+.3f}")

In [ ]:
plt.errorbar(range(len(tab)), tab.ATE, yerr=[tab.ATE - tab["IC inf"], tab["IC sup"] - tab.ATE], fmt="o", capsize=4)
plt.xticks(range(len(tab)), [c.replace("(+3 fugas)", "").replace("base + ", "+") for c in tab.covariables], rotation=25)
plt.ylabel("ATE estimado"); plt.title("Cada fuga mueve el efecto"); plt.show()

**Mira el resultado.** La métrica predictiva *sube* con las fugas. ¿Qué pasa con el efecto y por qué importa para decidir con la métrica?

**Respuesta:**

El R² mejora y el efecto se desplaza (o se hunde). **Optimizar la métrica predictiva no protege la validez causal**: con fugas la métrica premia justo lo que invalida el efecto.



## 4. Un detector por contenido
Sin metadatos, ¿qué firma deja cada fuga en los datos?
- una variable **derivada de $Y$** se parece demasiado a $Y$ (|corr(x, y)| muy alta);
- una variable **causada por $T$** se parece demasiado a $T$ (|corr(x, t)| muy alta, más que lo que produce la confusión).

Escribe `detectar_fuga(df, y, t, umbral_y, umbral_t)` que devuelva los nombres sospechosos. Evalúa luego **precisión y exhaustividad** (¿encuentra las 3? ¿marca alguna limpia?).

In [ ]:
# ✏️ Completa tú — detectar_fuga
# Pista: np.corrcoef(df[c], y)[0, 1]
def detectar_fuga(df, y, t, umbral_y=0.75, umbral_t=0.5):
    """Lista de columnas de df sospechosas de fuga (derivadas de y o causadas por t)."""
    raise NotImplementedError("para cada columna calcula |corr| con y y con t y marca si pasa algún umbral")

In [ ]:
corr_y = df.apply(lambda c: abs(np.corrcoef(c, y)[0, 1])).sort_values(ascending=False)
print("|corr con y| de las 6 columnas más correlacionadas (elige el umbral mirando el salto):")
print(corr_y.head(6).round(2).to_string())
marcadas = detectar_fuga(df, y, t)
tp = len(set(marcadas) & set(FUGAS)); fp = len(set(marcadas) - set(FUGAS))
print("marcadas:", marcadas)
print(f"exhaustividad = {tp}/{len(FUGAS)} · falsos positivos (covariables limpias marcadas) = {fp}")
lab_check("detector encuentra las fugas", lambda: set(FUGAS) <= set(detectar_fuga(df, y, t)), [((), True, 0)])

**Mira el resultado.** ¿Qué fuga podría escapar a este detector aunque el nuestro las encontró todas? ¿Qué debe acompañarlo siempre?

**Respuesta:**

Una fuga **débil** (una covariable posterior al tratamiento con poca correlación con T o Y) pasa cualquier umbral razonable. Por eso el detector complementa, **no sustituye**, la auditoría por metadatos y el conocimiento de cuándo se mide cada columna.



## 5. La media móvil con fuga (ejercicio del capítulo)
Con series temporales, una media móvil **centrada** usa valores *futuros*; una **retrasada** solo el pasado. Ambas parecen «una media móvil»; solo una es legítima.

In [ ]:
r = np.random.default_rng(0)
serie = pd.Series(np.cumsum(r.normal(size=600)))
centrada = serie.rolling(7, center=True).mean()           # usa t+1, t+2, t+3  → FUGA
retrasada = serie.shift(1).rolling(7).mean()              # solo t-1 … t-7    → legítima
ok = centrada.notna() & retrasada.notna()
err = lambda f: np.sqrt(np.mean((serie[ok] - f[ok]) ** 2))
print(f"error de «predecir» la serie con la media centrada  : {err(centrada):.3f}")
print(f"error de predecir con la media retrasada            : {err(retrasada):.3f}")

## 6. Supuesto y bitácora

In [ ]:
fila = bitacora(
    pregunta="¿Ha entrado en las covariables alguna variable posterior al tratamiento o derivada de Y?",
    metodo="Auditoría por metadatos + detector por contenido; contraste del ATE con y sin las covariables sospechosas",
    estimando="ATE (OLS con controles) con X_base y con X_ampliada",
    numero_ic=f"X_base: {tab.ATE[0]:.2f}; X_ampliada: {tab.ATE[1]:.2f}; ΔATE = {e1 - e0:+.2f}",
    supuesto=supuesto("observable", "medido", f"{tp}/{len(FUGAS)} fugas halladas; ninguna covariable posterior al tratamiento en X_base"),
    decision="continuar", lab="L08")

---
## Qué queda para el libro
Ejercicios del libro cubiertos: clasificar variables por momento de medición, la media móvil con fuga y comparar IC con y sin las covariables sospechosas. Queda para casa: auditar ≥ 6 covariables de tu propio proyecto.

*Material formativo (sin nota). Los datos son sintéticos: ver la ficha del escenario.*